# 8j — Preliminary composable forecast (four ways), TWO-STAGE cut

Implements `inst/1a_preliminary_framework_plan.md` + `inst/1c`/`inst/1d`/`inst/1e`, and the
**two-stage cut inference** of `inst/4_cut_Bayes.md`: a renewal / next-generation-matrix model
driven by **age-pair contact-degree distributions**, scored **four ways** — the 2×2 grid of
{unweighted **NegBin**, weighted **Hurdle-Weibull**} degree models × {**Mean**, **Neighbourhood**} NGM.

**Cut inference.** The former joint fit is split: **Stage 1** fits the contact-degree GP alone
(`model_degree`), and **Stage 2** fits the infection/renewal block (`model_transmission`)
conditioning on a *fixed* contact matrix drawn from Stage 1. Stage-1 uncertainty is propagated by
imputing **100** Stage-1 posterior draws, re-fitting Stage 2 for each (keeping **100** draws), and
**pooling** the 100×100 = **10 000** infection draws as the predictive used for WIS.

The contact **mean** is estimated **per week** with **structural reciprocity**
(`log μ_{i→j} = r + log Nⱼ`) and **separable spatio-temporal-GP smoothing** across the age-pair
grid *and over weeks* (inst/1e, §5). The NGM uses the **per-contact secondary attack rate γ_SAR**
with **un-normalised** C* (the `-gnorm` C*/S̄ decoupling was reverted). Forecasts use the
**contact-updated iterate** over origins × 4 horizons; **WIS** is on a **log scale**, by horizon.

Stage 1 uses **Pathfinder.jl** (or NUTS via `STAGE1_USE_NUTS`); Stage 2 uses Pathfinder.

> **This notebook does FITTING ONLY.** It fits and caches both stages' artefacts (the slow part).
> Forecast assembly, WIS scoring, and diagnostics moved to `9j_forecast_diagnostics.ipynb`, which
> reloads these files.

In [1]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

# Stage-1 (contact-degree GP) sampler. false ⇒ Pathfinder (preliminary; the choice for this run);
# true ⇒ Turing NUTS (slower, more faithful GP posterior — the intended later switch). Stage 2
# (infection) is always Pathfinder (100 fast fits per Stage-1 draw). See inst/4_cut_Bayes.md.
STAGE1_USE_NUTS = false

false

## §1 Window, infection/antibody data, and age-pair degree data

In [2]:
# `constant_contacts = false` ⇒ contact degree estimated PER WEEK, temporally smoothed by a
# separable spatio-temporal GP (shared ρ_diag/ρ_gap/ρ_time, η, σ_c; scalar intercept c +
# temporal-level GP cₜ = c + σ_c·(Lt·z_c) + matrix-normal field η·Lp·z·Ltᵀ). The renewal NGM
# then varies in time through contacts as well as antibody: N(t) uses that week's C*ₜ.
# (Set true for the pooled one-C*-per-window preliminary.)
# `stage1_use_nuts` selects the Stage-1 (contact-degree) sampler; Stage 2 is always Pathfinder.
cfg  = FrameworkConfig(constant_contacts = false, stage1_use_nuts = STAGE1_USE_NUTS)
grid = cis_age_grid()

# Read the CoMix contact data AND the inc2prev infection/antibody estimates ONCE and reuse them
# across every window (avoids re-reading/re-joining the full Arrow and re-parsing the estimates
# CSV per origin×horizon). Then roll the forecast origin over the whole period the current
# datasets support ("available period"): each origin needs a 12-week fit/lag window back to the
# first inc2prev week, and contact data out to origin+4 for the contact-updated iterate
# (horizon-h contacts observed at t₀+h). `available_forecast_origins` derives the range.
raw  = load_raw_contact_inputs()
inf  = load_raw_infection_inputs()          # (; df, tmap) — read estimates_age_ab.csv once
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw)
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

println("contact data span  : ", extrema(skipmissing(raw.craw.date)))
println("forecast origins   : ", length(wins), " weekly, ",
        first(FORECAST_ORIGINS), " … ", last(FORECAST_ORIGINS))
let w = wins[1], wd0 = load_window_data(wins[1], inf.df, inf.tmap; grid = grid)
    println("origin[1] fit weeks: ", w.fit_weeks[1], " … ", w.fit_weeks[end])
    println("weekly infections @ origin[1] (age): ", round.(wd0.I_mean[:, end]; digits = 0))
end

contact data span  : (Date("2020-03-23"), Date("2021-06-23"))
forecast origins   : 32 weekly, 2020-10-18 … 2021-05-23
origin[1] fit weeks: 2020-08-30 … 2020-10-18
weekly infections @ origin[1] (age): [99876.0, 64778.0, 116703.0, 89814.0, 124356.0, 117410.0, 41714.0]


In [3]:
# Fit config: the four combos and the parallel-fit concurrency (CPU- and memory-balanced).
combos = [(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                    for nb in (MeanNGM(), NeighbourhoodDegreeNGM())]
MAX_FIT_CONCURRENCY = fit_concurrency()          # min(threads, cores−1, RAM-budget)
if Threads.nthreads() == 1
    @warn "Julia has 1 thread — pre-fit runs sequentially. Start with JULIA_NUM_THREADS>1 " *
          "(e.g. $(max(1, Sys.CPU_THREADS - 1))) for parallel fitting."
end
println("combos = ", length(combos), " | fit concurrency = ", MAX_FIT_CONCURRENCY,
        " | total fits = ", length(wins) * length(combos) * length(cfg.horizons),
        " (cached ones are skipped)")

combos = 4 | fit concurrency = 9 | total fits = 512 (cached ones are skipped)


## §2 Roll over the available period — two-stage fit, forecast 1–4 weeks ahead

For **each weekly origin** across the available period the four combos are fit in two stages and
forecast. The contact **mean** is a **reciprocity-structural, GP-smoothed** field
(`log μ_{i→j}=r+log Nⱼ` ⟹ exact reciprocity), estimated per week over the fit window **and the
forecast weeks** — the horizon-`h` contact window ends at `t₀+h`, separately per horizon (inst/1d).

**Stage 1 (`prefit_stage1!`)** fits that GP once per (degree × origin × horizon) — it is
NGM-independent, so one fit serves both builders. **Stage 2 (`prefit_stage2!`)** then, per
(degree × ngm × origin × horizon), imputes 100 Stage-1 posterior draws, re-fits the infection block
`model_transmission` conditioning on each (Pathfinder), and pools 100×100 = 10 000 infection draws.
Forecasting is the **contact-updated iterate**: per origin t₀ and horizon `h` the NGM uses the
Stage-1 draw's `t₀+h` C* and the paired Stage-2 infection draw, and one renewal step is taken.

Origins run **sequentially** (bounded memory); within an origin Stage-1 fits fan out over threads
and each Stage-2 cell fans out its 100 per-draw fits (`MAX_FIT_CONCURRENCY`). Both stages are
cached (`8j_s1_*` / `8j_s2_*` under `../dt_intermediate`), so the run is **resumable** — a re-run
reloads finished files and only fits what's missing.

In [ ]:
# Parallel, resumable TWO-STAGE PRE-FIT — the cut inference (inst/4_cut_Bayes.md).
#   Stage 1: fit the contact-degree GP once per (degree × origin × horizon) — NGM-independent —
#            and cache it to ../dt_intermediate/8j_s1_<degree>_<contacts>_<origin>_h<h>.jld2.
#   Stage 2: for each (degree × ngm × origin × horizon), impute 100 Stage-1 posterior draws, re-fit
#            the infection block conditioning on each (Pathfinder), keep 100 draws, and cache the
#            100×100 = 10_000 pooled infection draws to 8j_s2_<degree>_<ngm>_<contacts>_<origin>_h<h>.jld2.
# Origins run sequentially (bounded memory); within an origin Stage-1 fits fan out over threads and
# each Stage-2 cell fans out its 100 per-draw fits. Cached files are skipped ⇒ resumable. Forecast
# assembly, WIS scoring, and diagnostics live in 9j_forecast_diagnostics.ipynb (run this first).

# One origin's datasets: window infection/antibody + the 4 contact/degree windows. Pure &
# deterministic given the shared read-only `inf`/`raw` reads, so it is safe to call from the
# per-origin prefit driver.
build_origin_data(oi, win_o) = (
    load_window_data(win_o, inf.df, inf.tmap; grid = grid),                     # reuse the single CSV read
    [prepare_degree_data(
         WeeklyWindow(win_o.origin + Day(7 * h);
                      n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons),
         cfg; grid = grid, setting = :all,
         df_part_raw = raw.df_part, craw_raw = raw.craw)                        # reuse the single Arrow read
     for h in cfg.horizons])

t0  = time()
res = prefit_two_stage!(combos, wins, cfg;
          data_provider = build_origin_data,
          save_dir = "../dt_intermediate", max_concurrent = MAX_FIT_CONCURRENCY)
println("two-stage pre-fit: Stage 1 ", res.stage1.fitted, " fitted / ", res.stage1.skipped,
        " skipped; Stage 2 ", res.stage2.fitted, " fitted / ", res.stage2.skipped,
        " skipped in ", round(Int, time() - t0), "s")

# Verifiable tail: count how many of this run's Stage-2 pooled files are present.
s2_paths = ["../dt_intermediate/8j_s2_$(degree_label(dm))_$(ngm_label(nb))_" *
            "$(contacts_label(cfg))_$(win.origin)_h$(h).jld2"
            for win in wins, (dm, nb) in combos, h in cfg.horizons]
n_have, n_expect = count(isfile, s2_paths), length(s2_paths)
println("pooled Stage-2 files: $n_have / $n_expect present under ../dt_intermediate ",
        "(", contacts_label(cfg), " contacts) — feed 9j_forecast_diagnostics.ipynb")